# 

# Generalization Datasets

- Classificaiton
  - tox21
  - toxcast
  - muv
  - pcba
- Regression
  - hopv - homo, lumo
  - zinc15 - logp
  - freesolv - hydration free energy
- Rxn
  - open reaction database
    - presto dataset( not compare with presto, because we assume OOD comparison)
- M2T
  - hanbum's dataset
- T2M
  - hanbum's dataset

# Check dataset availability

In [20]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem


def mol2graph(mol):
    """
    Converts SMILES string to graph Data object
    :input: SMILES string (str)
    :return: graph object
    """
    # atoms
    atom_features_list = []
    for atom in mol.GetAtoms():
        atom_features_list.append(atom_to_feature_vector(atom))
    x = np.array(atom_features_list, dtype = np.int64)

    # bonds
    num_bond_features = 3  # bond type, bond stereo, is_conjugated
    if len(mol.GetBonds()) > 0: # mol has bonds
        edges_list = []
        edge_features_list = []
        for bond in mol.GetBonds():
            i = bond.GetBeginAtomIdx()
            j = bond.GetEndAtomIdx()

            edge_feature = bond_to_feature_vector(bond)

            # add edges in both directions
            edges_list.append((i, j))
            edge_features_list.append(edge_feature)
            edges_list.append((j, i))
            edge_features_list.append(edge_feature)

        # data.edge_index: Graph connectivity in COO format with shape [2, num_edges]
        edge_index = np.array(edges_list, dtype = np.int64).T

        # data.edge_attr: Edge feature matrix with shape [num_edges, num_edge_features]
        edge_attr = np.array(edge_features_list, dtype = np.int64)

    else:   # mol has no bonds
        edge_index = np.empty((2, 0), dtype = np.int64)
        edge_attr = np.empty((0, num_bond_features), dtype = np.int64)

    graph = dict()
    graph['edge_index'] = edge_index
    graph['edge_feat'] = edge_attr
    graph['node_feat'] = x
    graph['num_nodes'] = len(x)

    return graph 

from rdkit import Chem
import selfies as sf
from download_dataset import wrap_label

system_prompt = "You are a helpful assistant for molecular chemistry, to address tasks including molecular property classification, molecular property regression, chemical reaction prediction, molecule captioning, molecule generation."

def prepare_data_instance(
        mol,
        label,
        task,
        instruction_templates,
        system_prompt,
        mol_token="<mol>",
        num_query_tokens=32,
):

    label = wrap_label(label, task=task)
    input_prompt = np.random.choice(instruction_templates).item()
    assert "<INPUT>" in input_prompt, f"llm_prompt should contain <INPUT>"
    graph_sequence = "<GRAPH>" + mol_token * num_query_tokens + "</GRAPH>"

    if "reagent_prediction" in task:
        smiles = Chem.MolToSmiles(mol[0])
        selfies = sf.encoder(smiles)
        input_mol_string = "<SELFIES> " + selfies + " </SELFIES>"
        input_mol_string_graph = input_mol_string + graph_sequence

        additional_smiles = Chem.MolToSmiles(mol[1])
        additional_selfies = sf.encoder(additional_smiles)
        additional_input_mol_string = "<SELFIES> " + additional_selfies + " </SELFIES>"
        additional_input_mol_string_graph = additional_input_mol_string + graph_sequence

        input_mol_string = input_mol_string + "|>>|" + additional_input_mol_string
        input_mol_string_graph = input_mol_string_graph + "|>>|" + additional_input_mol_string_graph
        
        graph = mol2graph(mol[0])
        additional_graph = mol2graph(mol[1])
    else:
        assert isinstance(mol, Chem.Mol), f"mol should be a RDKit Mol object, but got {type(mol)}"
        mol = mol
        smiles = Chem.MolToSmiles(mol)
        selfies = sf.encoder(smiles)
        input_mol_string = "<SELFIES> " + selfies + " </SELFIES>"
        input_mol_string_graph = input_mol_string + graph_sequence

        graph = mol2graph(mol)
        additional_graph = graph
    
    input_prompt = input_prompt.replace("<INPUT>", input_mol_string_graph)

    formatted_prompt_text = "<s>[INST] " + system_prompt + " \n\n" + input_prompt + " [INST]"
    formatted_target_text = label + " </s>"


    data = {
        "task": task,
        "x": graph['node_feat'],
        "edge_index": graph['edge_index'],
        "edge_attr": graph['edge_feat'],
        "additional_x": additional_graph['node_feat'],
        "additional_edge_index": additional_graph['edge_index'],
        "additional_edge_attr": additional_graph['edge_feat'],
        "input_mol_string": input_mol_string,
        "prompt_text": formatted_prompt_text,
        "target_text": formatted_target_text,
    }
    return data


def get_data_list(
        list_mol, list_label, task, instruction_templates, system_prompt
):
    list_data = []
    iter_bar = tqdm(range(len(list_mol)))

    for i in iter_bar:
        data = prepare_data_instance(
        mol=list_mol[i],
        label=list_label[i], 
        task=task,
        instruction_templates=instruction_templates,
        system_prompt=system_prompt
    )  
        list_data.append(data)
    return list_data


In [21]:
# alchemy dataset
import pandas as pd
import os

data_dir = "/data/data/Alchemy-v20191129"
csv_label_path = f"{data_dir}/final_version.csv"

label = pd.read_csv(csv_label_path)

list_atom9 = os.listdir(f"{data_dir}/atom_9")
list_atom10 = os.listdir(f"{data_dir}/atom_10")
list_atom11 = os.listdir(f"{data_dir}/atom_11")
list_atom12 = os.listdir(f"{data_dir}/atom_12")

print(len(list_atom9), len(list_atom10), len(list_atom11), len(list_atom12), len(label))

np.random.shuffle(list_atom9)
np.random.shuffle(list_atom10)
np.random.shuffle(list_atom11)
np.random.shuffle(list_atom12)

num_sample = 250
list_atom9 = list_atom9[:num_sample]
list_atom10 = list_atom10[:num_sample]
list_atom11 = list_atom11[:num_sample]
list_atom12 = list_atom12[:num_sample]


35855 119661 37732 9331 202579


In [22]:
# get columns

label.columns

Index(['gdb_idx', 'atom number', 'zpve\n(Ha, zero point vibrational energy)',
       'Cv\n(cal/molK, heat capacity at 298.15 K)', 'gap\n(Ha, LUMO-HOMO)',
       'G\n(Ha, Free energy at 298.15 K)', 'HOMO\n(Ha, energy of HOMO)',
       'U\n(Ha, internal energy at 298.15 K)',
       'alpha\n(a_0^3, Isotropic polarizability)',
       'U0\n(Ha, internal energy at 0 K)', 'H\n(Ha, enthalpy at 298.15 K)',
       'LUMO\n(Ha, energy of LUMO)', 'mu\n(D, dipole moment)',
       'R2\n(a_0^2, electronic spatial extent)'],
      dtype='object')

In [23]:
def get_atom_dict(
        list_alchemy_mol,
        dir_name,
        data_dir="/data/data/Alchemy-v20191129"
):
    list_mol = []
    list_label_homo = []
    list_label_lumo = []
    list_label_gap = []
    omitted_idx = []

    for i in range(len(list_alchemy_mol)):
        gdb_idx = list_alchemy_mol[i].split(".")[0]
        sdf_path = f"{data_dir}/{dir_name}/{list_alchemy_mol[i]}"
        try:
            mol = Chem.SDMolSupplier(sdf_path)
            list_mol.append(mol)
            label_idx = label[label['gdb_idx'] == int(gdb_idx)].index
            homo_value = label['HOMO\n(Ha, energy of HOMO)'][label_idx].item()
            lumo_value = label['LUMO\n(Ha, energy of LUMO)'][label_idx].item()
            gap_value = label['gap\n(Ha, LUMO-HOMO)'][label_idx].item()
            list_label_homo.append(homo_value)
            list_label_lumo.append(lumo_value)
            list_label_gap.append(gap_value)
        except:
            omitted_idx.append(i)

    print(len(list_mol), len(omitted_idx), len(list_label_homo), len(list_label_lumo), len(list_label_gap))
    return {
        'mol': list_mol,
        'homo': list_label_homo,
        'lumo': list_label_lumo,
        'gap': list_label_gap,
    }

In [24]:
atom9_dict = get_atom_dict(
    list_alchemy_mol=list_atom9,
    dir_name="atom_9"
)
atom10_dict = get_atom_dict(
    list_alchemy_mol=list_atom10,
    dir_name="atom_10"
)
atom11_dict = get_atom_dict(
    list_alchemy_mol=list_atom11,
    dir_name="atom_11"
)
atom12_dict = get_atom_dict(
    list_alchemy_mol=list_atom12,
    dir_name="atom_12"
)

250 0 250 250 250
250 0 250 250 250
250 0 250 250 250
250 0 250 250 250


In [25]:
# concat atom dicts from 9, 10, 11, 12
list_alchemy_te_mol = atom9_dict['mol'] + atom10_dict['mol'] + atom11_dict['mol'] + atom12_dict['mol']
list_alchemy_te_mol = [i[0] for i in list_alchemy_te_mol]
list_alchemy_te_label_homo = atom9_dict['homo'] + atom10_dict['homo'] + atom11_dict['homo'] + atom12_dict['homo']
list_alchemy_te_label_lumo = atom9_dict['lumo'] + atom10_dict['lumo'] + atom11_dict['lumo'] + atom12_dict['lumo']
list_alchemy_te_label_homo_lumo_gap = atom9_dict['gap'] + atom10_dict['gap'] + atom11_dict['gap'] + atom12_dict['gap']

In [26]:
len(list_alchemy_te_mol), len(list_alchemy_te_label_homo), len(list_alchemy_te_label_lumo), len(list_alchemy_te_label_homo_lumo_gap)

(1000, 1000, 1000, 1000)

In [27]:
list_alchemy_homo_data = get_data_list(
    list_mol=list_alchemy_te_mol,
    list_label=list_alchemy_te_label_homo,
    task="alchemy_homo",
    instruction_templates=instructions_smol.qm9_homo,
    system_prompt=system_prompt
)

list_alchemy_lumo_data = get_data_list(
    list_mol=list_alchemy_te_mol,
    list_label=list_alchemy_te_label_lumo,
    task="alchemy_lumo",
    instruction_templates=instructions_smol.qm9_lumo,
    system_prompt=system_prompt
)

list_alchemy_homo_lumo_gap_data = get_data_list(
    list_mol=list_alchemy_te_mol,
    list_label=list_alchemy_te_label_homo_lumo_gap,
    task="alchemy_homo_lumo_gap",
    instruction_templates=instructions_smol.qm9_homo_lumo_gap,
    system_prompt=system_prompt
)

100%|██████████| 1000/1000 [00:01<00:00, 871.19it/s]


In [33]:
alchemy_te_data = list_alchemy_homo_data + \
                list_alchemy_lumo_data + \
                    list_alchemy_homo_lumo_gap_data
alchemy_te_dataset = datasets.Dataset.from_list(alchemy_te_data)
alchemy_te_dataset.save_to_disk(f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_alchemy1k_0211")

Saving the dataset (1/1 shards): 100%|██████████| 3000/3000 [00:00<00:00, 64380.83 examples/s]


In [9]:
mol_instruction_dataset = load_dataset(
    "zjunlp/Mol-Instructions",
    "Molecule-oriented Instructions",
    trust_remote_code=True,
)
qm9_data = mol_instruction_dataset['property_prediction']
qm9_homo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo
            )
qm9_lumo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_lumo
            )
qm9_homo_lumo_gap_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo_lumo_gap
            )

Filter: 100%|██████████| 362100/362100 [00:12<00:00, 29044.35 examples/s]


In [10]:
len(qm9_homo_data), len(qm9_lumo_data), len(qm9_homo_lumo_gap_data)

(120746, 120753, 120601)

In [11]:
qm9_homo_data

Dataset({
    features: ['instruction', 'input', 'output', 'metadata'],
    num_rows: 120746
})

In [29]:
def get_qm9_data_list(
        data,
        task,
        instruction_templates,
        system_prompt
):
    list_tr_mol = []
    list_tr_label = []

    list_te_mol = []
    list_te_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(data)))
    for i in iter_bar:
        data_instance = data[i]
        selfies = data_instance['input']
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_tr_label.append(label)
            list_tr_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_te_label.append(label)
            list_te_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError

    list_tr_data = get_data_list(
        list_mol=list_tr_mol,
        list_label=list_tr_label,
        task=task,
        instruction_templates=instruction_templates,
        system_prompt=system_prompt
    )
    list_te_data = get_data_list(
        list_mol=list_te_mol,
        list_label=list_te_label,
        task=task,
        instruction_templates=instruction_templates,
        system_prompt=system_prompt
    )
    print(len(list_tr_data), len(list_te_data))
    return list_tr_data, list_te_data

In [30]:
list_qm9_homo_tr_data, list_qm9_homo_te_data = get_qm9_data_list(
    data=qm9_homo_data,
    instruction_templates=instructions_smol.qm9_homo,
    task="qm9_homo",
    system_prompt=system_prompt
)

100%|██████████| 684/684 [00:00<00:00, 1002.58it/s]


In [31]:
list_qm9_lumo_tr_data, list_qm9_lumo_te_data = get_qm9_data_list(
    data=qm9_lumo_data,
    instruction_templates=instructions_smol.qm9_lumo,
    task="qm9_lumo",
    system_prompt=system_prompt
)

100%|██████████| 642/642 [00:00<00:00, 970.50it/s] 


In [32]:
list_qm9_homo_lumo_gap_tr_data, list_qm9_homo_lumo_gap_te_data = get_qm9_data_list(
    data=qm9_homo_lumo_gap_data,
    instruction_templates=instructions_smol.qm9_homo_lumo_gap,
    task="qm9_homo_lumo_gap",
    system_prompt=system_prompt
)

100%|██████████| 661/661 [00:00<00:00, 1025.28it/s]


In [34]:
augmented_qm9_te_data = list_qm9_homo_te_data \
    + list_qm9_lumo_te_data + \
        list_qm9_homo_lumo_gap_te_data + \
            list_alchemy_homo_data + \
                list_alchemy_lumo_data + \
                    list_alchemy_homo_lumo_gap_data
print(
    len(list_qm9_homo_te_data),
    len(list_qm9_lumo_te_data),
    len(list_qm9_homo_lumo_gap_te_data),
    len(list_alchemy_homo_data), 
    len(list_alchemy_lumo_data), 
    len(list_alchemy_homo_lumo_gap_data)
    )

qm9_te_dataset = datasets.Dataset.from_list(augmented_qm9_te_data)
qm9_te_dataset.save_to_disk(f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_0211")

684 642 661 1000 1000 1000


Saving the dataset (1/1 shards): 100%|██████████| 4987/4987 [00:00<00:00, 86759.75 examples/s]


In [35]:
augmented_qm9_tr_data = list_qm9_homo_tr_data + list_qm9_lumo_tr_data + list_qm9_homo_lumo_gap_tr_data
print(
    len(list_qm9_homo_tr_data),
    len(list_qm9_lumo_tr_data),
    len(list_qm9_homo_lumo_gap_tr_data),
    )

qm9_tr_dataset = datasets.Dataset.from_list(augmented_qm9_tr_data)
qm9_tr_dataset.save_to_disk(f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_0211")

120062 120111 119940


Saving the dataset (3/3 shards): 100%|██████████| 360113/360113 [00:10<00:00, 34413.16 examples/s] 
